In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, Optional

ROOT_DIR = Path("..").resolve()
sys.path.append(str(ROOT_DIR))

from src.utils.processing_utils import clip
from src.utils.debug import print_images_statistics
from src.utils.metrics import get_all_distortion_metrics
from src.utils.constants import EPS

# --- CONFIGURATION ---
# FPGA inference results directory.
RESULTS_DIR = ROOT_DIR / "results/fpga/compiled_models/ResSHyp-relu_s0_L500_pt/results"

# GPU training run directory (contains recon_*_linA.npy saved by CompareReconstructionToGT).
# Set to None to auto-derive from manifest.json in the compiled_models/<model>/ folder.
GPU_RUN_DIR: Optional[Path] = ROOT_DIR / "logs/train/sar_ddc/hyperprior/multiruns/2026-02-06_14-57-22/3"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"FPGA results directory not found: {RESULTS_DIR}")


In [ ]:
# 1. Print Execution Summary
metrics_path = RESULTS_DIR / "metrics.json"
print(metrics_path.resolve())
log_path = RESULTS_DIR / "inference.log"

# Check Log Header for Model Info
if log_path.exists():
    print("\n--- Run Info ---")
    with open(log_path, "r") as f:
        for line in f:
            if "Model" in line or "DPU" in line or "Inference" in line:
                print(line.strip())

# Print Metrics
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print("\n--- Final Metrics ---")
    print(json.dumps(metrics, indent=4))
else:
    print("\nmetrics.json not found in result directory.")

In [ ]:
recons_dir = RESULTS_DIR / "reconstructions_test_set"
vis_noisy = np.load(recons_dir / "vis_noisy.npy")
vis_recon = np.load(recons_dir / "vis_recon.npy")
vis_adam = np.load(recons_dir / "vis_adam.npy")
vis_merlin = np.load(recons_dir / "vis_merlin.npy")

print(f"Loaded {len(vis_noisy)} patches for visualization.")

# --- Visualization Logic (Inline to ensure correctness) ---
num_patches = len(vis_noisy)
N = min(5, num_patches)

fig, axes = plt.subplots(4, N, figsize=(4 * N, 16))
if N == 1:
    axes = axes.reshape(4, 1)

for i in range(N):
    # Clip individual patches (not the whole batch)
    noisy_disp = clip(vis_noisy[i])
    recon_disp = clip(vis_recon[i])
    adam_disp = clip(vis_adam[i])
    merlin_disp = clip(vis_merlin[i])

    # Row 0: Original Noisy (LogI)
    axes[0, i].imshow(noisy_disp, cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Noisy Input")

    # Row 1: Reconstruction (LogI)
    axes[1, i].imshow(recon_disp, cmap="gray")
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("Reconstruction")

    # Row 2: ADAM NOC GT (LogI)
    axes[2, i].imshow(adam_disp, cmap="gray")
    axes[2, i].axis("off")
    if i == 0:
        axes[2, i].set_title("ADAM NOC GT")

    # Row 3: MERLIN GT (LogI)
    axes[3, i].imshow(merlin_disp, cmap="gray")
    axes[3, i].axis("off")
    if i == 0:
        axes[3, i].set_title("MERLIN GT")

plt.tight_layout()
plt.show()
fig.savefig(RESULTS_DIR / "reconstructions_visualization.png")

In [ ]:
# ── Helper Functions ─────────────────────────────────────────────────────────

def _remap_to_local(path: Path, root_dir: Path) -> Path:
    """Remap a Docker/remote path to the local ROOT_DIR.

    Handles any path that contains a 'DDC_FPGA' component, e.g.:
      /workspace/DDC_FPGA/logs/...
      /mnt/vitisAI/Vitis-AI/DDC_FPGA/logs/...
    Both map to root_dir / logs / ...
    """
    parts = path.parts
    try:
        idx = parts.index("DDC_FPGA")
        return root_dir / Path(*parts[idx + 1:])
    except ValueError:
        return path  # No DDC_FPGA component found; return unchanged.


def get_gpu_run_dir(
    results_dir: Path,
    override: Optional[Path],
    root_dir: Path,
) -> Optional[Path]:
    """Return the GPU training run directory.

    Uses *override* directly if provided (caller supplies a local path).
    Otherwise reads *original_run_dir* from manifest.json located next to
    *results_dir* and applies Docker-to-local path remapping automatically.
    """
    if override is not None:
        return override

    manifest_path = results_dir.parent / "manifest.json"
    if not manifest_path.exists():
        print(f"[!] manifest.json not found at {manifest_path}")
        return None

    with open(manifest_path) as f:
        manifest = json.load(f)

    original_run_dir = manifest.get("original_run_dir", "")
    if not original_run_dir:
        print("[!] 'original_run_dir' not found in manifest.json")
        return None

    remapped = _remap_to_local(Path(original_run_dir), root_dir)
    print(f"GPU run dir (from manifest): {remapped.relative_to(root_dir)}")
    return remapped


def load_gpu_reconstruction(
    gpu_run_dir: Optional[Path],
    tile_name: str,
) -> Tuple[Optional[np.ndarray], Dict]:
    """Load GPU reconstruction from the training run directory.

    The CompareReconstructionToGT callback saves on on_test_end():
      recon_{tile_name}_linA.npy      — linear amplitude reconstruction
      recon_{tile_name}_metrics.json  — tile metrics (BPP, PSNR, …), when present

    Returns:
        gpu_linA : np.ndarray or None if the file is missing.
        metrics  : dict with tile metrics, or empty dict (BPP will be NaN).
    """
    if gpu_run_dir is None or not gpu_run_dir.exists():
        print(f"[!] GPU run dir not found: {gpu_run_dir}")
        return None, {}

    img_stem = f"recon_{tile_name}"
    linA_path = gpu_run_dir / f"{img_stem}_linA.npy"
    if not linA_path.exists():
        print(f"[!] GPU linA not found: {linA_path}")
        return None, {}

    gpu_linA = np.load(linA_path)
    print(f"GPU linA loaded: {linA_path.relative_to(ROOT_DIR)}  shape={gpu_linA.shape}")

    metrics: Dict = {}
    metrics_path = gpu_run_dir / f"{img_stem}_metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
        print(f"GPU tile metrics loaded: {metrics_path.name}")
    else:
        print(f"[~] No tile metrics file found ({img_stem}_metrics.json) — BPP will be unavailable.")

    return gpu_linA, metrics


In [ ]:
# ── 1. FPGA Reconstruction ───────────────────────────────────────────────────
all_recon_tiles = list(RESULTS_DIR.glob("*_recon_linA.npy"))
if not all_recon_tiles:
    raise FileNotFoundError(f"No *_recon_linA.npy found in {RESULTS_DIR}")

recon_tile_path = all_recon_tiles[0]
tile_name = recon_tile_path.name.replace("_recon_linA.npy", "")
print(f"Tile: {tile_name}")

fpga_linA = np.load(recon_tile_path)

fpga_metrics: Dict = {}
fpga_metrics_path = RESULTS_DIR / f"{tile_name}_metrics.json"
if fpga_metrics_path.exists():
    with open(fpga_metrics_path) as f:
        fpga_metrics = json.load(f)
    print(
        f"FPGA  bpp={fpga_metrics.get('bpp', float('nan')):.4f}  "
        f"psnr_MERLIN={fpga_metrics.get('psnr_MERLIN', float('nan')):.2f} dB"
    )
else:
    print(f"[!] FPGA tile metrics not found: {fpga_metrics_path.name}")

# ── 2. Reference Images ──────────────────────────────────────────────────────
tile_data_dir = ROOT_DIR / "data" / "visualization" / tile_name
if not tile_data_dir.exists():
    raise FileNotFoundError(f"Reference data not found: {tile_data_dir}")

noisy_linA      = np.load(tile_data_dir / "linA_Noisy.npy")
merlin_linA     = np.load(tile_data_dir / "linA_MERLIN.npy")
adam_noc_linA   = np.load(tile_data_dir / "linA_ADAM_NOC.npy")
merlin_dds_linA = np.load(tile_data_dir / "linA_MERLIN_DDS.npy")
print(f"Reference images loaded from: data/visualization/{tile_name}/")

# ── 3. GPU Reconstruction ────────────────────────────────────────────────────
gpu_run_dir = get_gpu_run_dir(RESULTS_DIR, GPU_RUN_DIR, ROOT_DIR)
gpu_linA, gpu_tile_metrics = load_gpu_reconstruction(gpu_run_dir, tile_name)

# Compute tile-level distortion metrics for GPU directly from the loaded arrays.
# BPP comes from the tile metrics JSON if available (saved by the callback on_test_end).
gpu_metrics: Dict = {}
if gpu_linA is not None:
    for ref_name, ref_arr in [
        ("MERLIN",     merlin_linA),
        ("ADAM-NOC",   adam_noc_linA),
        ("MERLIN_DDS", merlin_dds_linA),
    ]:
        m = get_all_distortion_metrics(gpu_linA, ref_arr)
        gpu_metrics[f"psnr_{ref_name}"] = m["psnr"]
        gpu_metrics[f"mse_{ref_name}"]  = m["mse"]
        gpu_metrics[f"ssim_{ref_name}"] = m["ssim"]
    gpu_metrics["bpp"]           = gpu_tile_metrics.get("bpp",           float("nan"))
    gpu_metrics["bpp_bitstream"] = gpu_tile_metrics.get("bpp_bitstream", float("nan"))
    _bpp_str = "N/A" if np.isnan(gpu_metrics["bpp"]) else f"{gpu_metrics['bpp']:.4f}"
    print(f"GPU   bpp={_bpp_str}  psnr_MERLIN={gpu_metrics['psnr_MERLIN']:.2f} dB")

# ── 4. Statistics Summary ─────────────────────────────────────────────────────
images_stats = {
    k: v for k, v in {
        "Noisy":       noisy_linA,
        "FPGA Recon":  fpga_linA,
        "GPU Recon":   gpu_linA,
        "MERLIN":      merlin_linA,
        "ADAM-NOC":    adam_noc_linA,
        "MERLIN_DDS":  merlin_dds_linA,
    }.items() if v is not None
}
print_images_statistics(
    images_stats,
    metrics=["min", "max", "mean", "std"],
    title="Linear Amplitude Statistics",
)

# ── 5. Overview Plot (6-panel) ────────────────────────────────────────────────
def _logI(linA: np.ndarray) -> np.ndarray:
    """Convert linear amplitude to clipped log-intensity for visualization."""
    return clip(2 * np.log(linA + EPS))

def _fmt(val: float, fmt: str = ".4f") -> str:
    """Format a float metric, returning 'N/A' for NaN."""
    return "N/A" if (isinstance(val, float) and np.isnan(val)) else format(val, fmt)

fpga_psnr = fpga_metrics.get("psnr_MERLIN", float("nan"))
fpga_bpp  = fpga_metrics.get("bpp",         float("nan"))
gpu_psnr  = gpu_metrics.get("psnr_MERLIN",  float("nan"))
gpu_bpp   = gpu_metrics.get("bpp",          float("nan"))

panels = [
    (noisy_linA,      "Noisy Input"),
    (adam_noc_linA,   "ADAM-NOC GT"),
    (merlin_linA,     "MERLIN GT"),
    (merlin_dds_linA, "MERLIN_DDS GT"),
    (fpga_linA,       f"FPGA Reconstruction\nBPP={_fmt(fpga_bpp)} | PSNR={_fmt(fpga_psnr, '.2f')} dB"),
    (gpu_linA,        f"GPU Reconstruction\nBPP={_fmt(gpu_bpp)} | PSNR={_fmt(gpu_psnr, '.2f')} dB"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (img, title) in zip(axes.flat, panels):
    if img is not None:
        im = ax.imshow(_logI(img), cmap="gray")
        fig.colorbar(im, ax=ax, shrink=0.7)
    else:
        ax.text(0.5, 0.5, "Not Available", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

fig.suptitle(f"Large Tile Comparison — {tile_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "tile_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Individual: FPGA Reconstruction ──────────────────────────────────────────
fpga_bpp_val  = fpga_metrics.get("bpp",         float("nan"))
fpga_psnr_val = fpga_metrics.get("psnr_MERLIN", float("nan"))
_bpp_s  = "N/A" if np.isnan(fpga_bpp_val)  else f"{fpga_bpp_val:.4f}"
_psnr_s = "N/A" if np.isnan(fpga_psnr_val) else f"{fpga_psnr_val:.2f} dB"

fig_fpga, ax_fpga = plt.subplots(figsize=(6, 6))
im_fpga = ax_fpga.imshow(clip(2 * np.log(fpga_linA + EPS)), cmap="gray")
ax_fpga.set_title(
    f"Reconstruction (FPGA ZCU102)\nBPP={_bpp_s} | PSNR(MERLIN)={_psnr_s}",
    fontsize=12, fontweight="bold",
)
# ax_fpga.axis("off")
fig_fpga.colorbar(im_fpga, ax=ax_fpga, shrink=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# ── Individual: GPU Reconstruction ───────────────────────────────────────────
if gpu_linA is not None:
    gpu_bpp_val  = gpu_metrics.get("bpp",         float("nan"))
    gpu_psnr_val = gpu_metrics.get("psnr_MERLIN",  float("nan"))
    _bpp_s  = "N/A" if np.isnan(gpu_bpp_val)  else f"{gpu_bpp_val:.4f}"
    _psnr_s = "N/A" if np.isnan(gpu_psnr_val) else f"{gpu_psnr_val:.2f} dB"

    fig_gpu, ax_gpu = plt.subplots(figsize=(6, 6))
    im_gpu = ax_gpu.imshow(clip(2 * np.log(gpu_linA + EPS)), cmap="gray")
    ax_gpu.set_title(
        f"Reconstruction (GPU: NVIDIA RTX A4000)\nBPP={_bpp_s} | PSNR(MERLIN)={_psnr_s}",
        fontsize=12, fontweight="bold",
    )
    ax_gpu.axis("off")
    fig_gpu.colorbar(im_gpu, ax=ax_gpu, shrink=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("[!] GPU reconstruction not available — check GPU_RUN_DIR in the config cell.")
